# Exercises — Chapter 13: LLM Limitations and Evaluation Rigor in Finance

Complete the exercises from Lecture 13.

In [ ]:
# Your code here

## Data Lab -- SEC EDGAR

Build a ground-truth numerical QA benchmark from EDGAR XBRL data and evaluate a retrieval system and a simulated LLM on it.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course your-email@example.com"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

### Exercise [B]: Numerical QA Benchmark Construction

In [ ]:
# --- Exercise [B]: Numerical QA Benchmark ---
BENCH_SPECS = [
    ("AAPL", "Revenues"),
    ("AAPL", "NetIncomeLoss"),
    ("AAPL", "Assets"),
    ("AAPL", "LongTermDebt"),
    ("AAPL", "StockholdersEquity"),
    ("MSFT", "Revenues"),
    ("MSFT", "NetIncomeLoss"),
    ("MSFT", "Assets"),
    ("MSFT", "LongTermDebt"),
    ("MSFT", "StockholdersEquity"),
    ("GOOGL", "Revenues"),
    ("GOOGL", "NetIncomeLoss"),
    ("GOOGL", "Assets"),
    ("GOOGL", "LongTermDebt"),
    ("GOOGL", "StockholdersEquity"),
]

qa_pairs = []
for ticker, concept in BENCH_SPECS:
    try:
        cik = get_cik(ticker)
        df  = get_annual_series(cik, concept).sort_values("date")
        row = df.iloc[-1]
        year = int(str(row["date"])[:4])
        val  = float(row[concept])
        label = concept.replace("NetIncomeLoss","Net Income").replace("StockholdersEquity","Stockholders Equity")
        qa_pairs.append({
            "ticker": ticker,
            "question": f"What was {ticker}'s {label} in fiscal year {year}?",
            "answer_usd": val,
            "answer_formatted": f"${val/1e9:.2f}B" if abs(val)>=1e9 else f"${val/1e6:.1f}M",
        })
    except Exception as e:
        print(f"{ticker}/{concept}: {e}")

for i, qa in enumerate(qa_pairs):
    print(f"{i+1:2d}. [{qa['ticker']}] {qa['question']}")
    print(f"    Answer: {qa['answer_formatted']}\n")
print(f"Total QA pairs: {len(qa_pairs)}")

### Exercise [I]: TF-IDF Retrieval Evaluation

In [ ]:
# --- Exercise [I]: TF-IDF Retrieval ---
# Build a TF-IDF index over AAPL's 5 most recent 10-K MD&A sections
cik_aapl = get_cik("AAPL")
subs_a   = get_submissions(cik_aapl)
fa       = subs_a["filings"]["recent"]
k10_aapl = [i for i, x in enumerate(fa["form"]) if x == "10-K"][:5]

docs, doc_years = [], []
for i in k10_aapl:
    acc  = fa["accessionNumber"][i].replace("-","")
    yr   = int(fa["filingDate"][i][:4])
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik_aapl.lstrip('0')}"
            f"/{acc}/{fa['primaryDocument'][i]}")
    time.sleep(0.15)
    html = requests.get(url, headers=EDGAR_HEADERS, timeout=30).text
    mda  = extract_mda(html)
    docs.append(mda); doc_years.append(yr)
    print(f"AAPL {yr}: {len(mda.split())} words")

# Build TF-IDF
def tokenise(text): return re.findall(r"[a-z]+", text.lower())
vocab = {}
for doc in docs:
    for w in set(tokenise(doc)):
        vocab[w] = vocab.get(w, 0) + 1
idf = {w: np.log(len(docs)/(c+1)) for w, c in vocab.items()}

def tfidf_vec(text):
    toks = tokenise(text)
    tf   = {}
    for w in toks: tf[w] = tf.get(w, 0) + 1/max(len(toks),1)
    return {w: tf[w]*idf.get(w, 0) for w in tf}

doc_vecs = [tfidf_vec(d) for d in docs]

def cosine(a, b):
    keys = set(a) & set(b)
    num  = sum(a[k]*b[k] for k in keys)
    da   = np.sqrt(sum(v**2 for v in a.values()))
    db   = np.sqrt(sum(v**2 for v in b.values()))
    return num / (da * db + 1e-12)

aapl_pairs = [qa for qa in qa_pairs if qa["ticker"] == "AAPL"]
hits = 0
for qa in aapl_pairs:
    qv     = tfidf_vec(qa["question"])
    scores = [cosine(qv, dv) for dv in doc_vecs]
    top    = doc_years[int(np.argmax(scores))]
    correct_yr = int(qa["question"].split("fiscal year ")[1].rstrip("?"))
    hit    = (top == correct_yr)
    hits  += int(hit)
    print(f"Q: {qa['question'][:60]}... -> top={top} (correct={correct_yr}) {'OK' if hit else 'X'}")

print(f"\nRecall@1 = {hits}/{len(aapl_pairs)} = {hits/max(len(aapl_pairs),1):.2f}")

### Exercise [A]: Hallucination Benchmark

In [ ]:
# --- Exercise [A]: Hallucination Benchmark ---
# Simulates realistic LLM numerical recall behaviour using a two-component mixture:
#   75% accurate recall  → uniform noise in [-10%, +10%]
#   25% hallucination    → large error in [25%, 90%], random direction
# This ensures some APE values exceed the 20% flagging threshold, making the
# benchmark informative (unlike a purely uniform ±15% noise which can never trigger it).
rng = np.random.default_rng(42)

results = []
for qa in qa_pairs:
    true_val = qa["answer_usd"]
    if rng.random() < 0.25:                            # hallucination event
        direction = rng.choice([-1.0, 1.0])
        magnitude = rng.uniform(0.25, 0.90)            # 25–90% absolute error
        noise = direction * magnitude
    else:                                              # accurate recall
        noise = rng.uniform(-0.10, 0.10)
    llm_val  = true_val * (1 + noise)
    ape      = abs(llm_val - true_val) / (abs(true_val) + 1e-9) * 100
    results.append({
        "ticker":   qa["ticker"],
        "concept":  qa["question"].split("'s ")[1].split(" in ")[0],
        "true_usd": true_val,
        "llm_usd":  llm_val,
        "ape":      ape,
        "flagged":  ape > 20,
    })

df_bench  = pd.DataFrame(results)
mape      = df_bench["ape"].mean()
n_flagged = df_bench["flagged"].sum()
print(f"MAPE: {mape:.1f}%")
print(f"Flagged (>20% error): {n_flagged}/{len(df_bench)}")
print()
print(df_bench[["ticker","concept","ape","flagged"]].to_string())

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["tomato" if f else "steelblue" for f in df_bench["flagged"]]
ax.bar(range(len(df_bench)), df_bench["ape"], color=colors)
ax.axhline(20, color="red", linestyle="--", linewidth=0.8, label="20% hallucination threshold")
ax.set_xticks(range(len(df_bench)))
ax.set_xticklabels(
    [f"{r['ticker'][:1]}-{r['concept'][:4]}" for _, r in df_bench.iterrows()],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylabel("APE (%)")
ax.set_title(
    "Simulated LLM Hallucination Benchmark\n"
    "(75% accurate recall ±10%, 25% hallucinated 25–90%)"
)
ax.legend()
plt.tight_layout()
plt.show()